
# PDF → JSON Document Tree

This notebook uploads a PDF **locally**, extracts its content, layout, images, links, annotations, fonts, page geometry, document metadata, and raw PDF-level information, and produces a single JSON tree.

### Output

The main output is:

```text
PDF
└── document
    ├── metadata
    ├── pages[]
    │   ├── metadata
    │   ├── dimensions
    │   ├── content[]
    │   │   ├── text / spans
    │   │   ├── images
    │   │   ├── drawings
    │   │   └── other blocks
    │   ├── links[]
    │   └── annotations[]
    ├── images[]
    ├── fonts[]
    └── statistics
```

The JSON is deliberately **information-preserving rather than presentation-oriented**. It keeps coordinates, page numbers, typography, image data, and source references so a later renderer can reconstruct the document as HTML.


In [1]:

# Install dependencies
%pip install -q pymupdf pillow


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:

import json
import base64
import hashlib
import io
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import fitz  # PyMuPDF
from PIL import Image
from IPython.display import display, HTML, JSON
# from google.colab import files

print("PyMuPDF:", fitz.__doc__.split()[1] if fitz.__doc__ else "unknown")


PyMuPDF: 1.26.5:



## 1. Upload the PDF

Run the cell below and select a PDF from your computer.


In [14]:

# uploaded = files.upload()

pdf_names = ["Statistics.pdf"]  # list(uploaded.keys())

if not pdf_names:
    raise ValueError("Please upload a PDF file.")

pdf_path = Path(pdf_names[0])

print(f"Selected: {pdf_path}")
print(f"Size: {pdf_path.stat().st_size:,} bytes")


Selected: Statistics.pdf
Size: 302,769 bytes


In [15]:

# Open the PDF
doc = fitz.open(pdf_path)

print("Pages:", len(doc))
print("PDF metadata:")
for key, value in doc.metadata.items():
    print(f"  {key}: {value}")


Pages: 4
PDF metadata:
  format: PDF 1.3
  title: 
  author: 
  subject: 
  keywords: 
  creator: 
  producer: macOS Version 26.5.1 (Build 25F80) Quartz PDFContext
  creationDate: D:20260807103454Z00'00'
  modDate: D:20260807103454Z00'00'
  trapped: 
  encryption: None



## 2. Helper functions

These functions normalize PyMuPDF objects into JSON-safe values and preserve the original PDF coordinates.

Coordinates are kept in PDF page space:

- origin: top-left
- x increases to the right
- y increases downward
- units: PDF points


In [16]:

def json_safe(value):
    """Convert common PDF/PyMuPDF/Python values into JSON-safe values."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, bytes):
        return {
            "encoding": "base64",
            "data": base64.b64encode(value).decode("ascii")
        }

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    # PyMuPDF Rect / Point / Matrix / Quad-like objects
    attrs = {}
    for attr in ("x0", "y0", "x1", "y1", "x", "y", "width", "height",
                 "a", "b", "c", "d", "e", "f"):
        if hasattr(value, attr):
            try:
                attrs[attr] = float(getattr(value, attr))
            except Exception:
                pass

    if attrs:
        return attrs

    try:
        return str(value)
    except Exception:
        return repr(value)


def rect_dict(rect):
    return {
        "x0": float(rect.x0),
        "y0": float(rect.y0),
        "x1": float(rect.x1),
        "y1": float(rect.y1),
        "width": float(rect.width),
        "height": float(rect.height),
    }


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def data_uri(data, mime_type):
    return f"data:{mime_type};base64,{base64.b64encode(data).decode('ascii')}"


def normalize_text(text):
    if text is None:
        return ""
    return text.replace("\x00", "").replace("\r\n", "\n").replace("\r", "\n")


def infer_image_mime(ext):
    ext = (ext or "").lower().lstrip(".")
    mapping = {
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "png": "image/png",
        "gif": "image/gif",
        "bmp": "image/bmp",
        "tif": "image/tiff",
        "tiff": "image/tiff",
        "jp2": "image/jp2",
        "webp": "image/webp",
    }
    return mapping.get(ext, "application/octet-stream")



## 3. Extract document metadata

This captures both standard PDF metadata and additional document-level information exposed by PyMuPDF.


In [17]:

def extract_document_metadata(doc):
    metadata = {
        "pdf_metadata": json_safe(doc.metadata),
        "page_count": len(doc),
        "is_encrypted": bool(doc.is_encrypted),
        "needs_pass": bool(doc.needs_pass),
        "permissions": json_safe(doc.permissions),
        "language": getattr(doc, "language", None),
        "metadata_extracted_at": datetime.now(timezone.utc).isoformat(),
    }

    # These are useful when available in the installed PyMuPDF version.
    for attr in ("metadata_xml", "xml_metadata"):
        if hasattr(doc, attr):
            try:
                value = getattr(doc, attr)
                metadata[attr] = json_safe(value() if callable(value) else value)
            except Exception as exc:
                metadata[f"{attr}_error"] = str(exc)

    return metadata

document_metadata = extract_document_metadata(doc)
JSON(document_metadata)


<IPython.core.display.JSON object>


## 4. Extract images

Every embedded image is exported as binary data and represented in the JSON with:

- stable image ID
- page reference
- xref
- original dimensions
- colorspace
- bits per component
- extension / MIME type
- byte size
- SHA-256 hash
- base64 data URI

The page tree also contains references to these images.


In [18]:

def extract_images(doc):
    images = []
    image_id_by_xref = {}

    for page_number, page in enumerate(doc, start=1):
        for image_index, image_info in enumerate(page.get_images(full=True), start=1):
            xref = int(image_info[0])

            if xref in image_id_by_xref:
                continue

            try:
                extracted = doc.extract_image(xref)
                image_bytes = extracted["image"]
                ext = extracted.get("ext", "bin")
                mime = infer_image_mime(ext)

                image_id = f"img_{len(images) + 1:05d}"
                image_id_by_xref[xref] = image_id

                images.append({
                    "id": image_id,
                    "xref": xref,
                    "first_seen_on_page": page_number,
                    "source_index": image_index,
                    "extension": ext,
                    "mime_type": mime,
                    "width": extracted.get("width"),
                    "height": extracted.get("height"),
                    "colorspace": extracted.get("colorspace"),
                    "bits_per_component": extracted.get("bpc"),
                    "size_bytes": len(image_bytes),
                    "sha256": sha256_bytes(image_bytes),
                    "data_base64": base64.b64encode(image_bytes).decode("ascii"),
                    "data_uri": data_uri(image_bytes, mime),
                })
            except Exception as exc:
                images.append({
                    "id": f"img_error_{len(images) + 1:05d}",
                    "xref": xref,
                    "error": str(exc),
                })

    return images, image_id_by_xref

images, image_id_by_xref = extract_images(doc)

print(f"Unique embedded images: {len(images)}")


Unique embedded images: 2



## 5. Extract page content

Each page is represented as an ordered tree of blocks.

For text, the hierarchy is:

```text
block
└── lines[]
    └── spans[]
        └── characters/text
```

This preserves typography and positioning rather than reducing the PDF to plain text.


In [19]:

def extract_text_block(block, page_number, block_index):
    # PyMuPDF dict blocks typically contain bbox, block_no, type, lines.
    result = {
        "id": f"p{page_number:04d}_block{block_index:04d}",
        "type": "text",
        "bbox": json_safe(block.get("bbox")),
        "block_no": block.get("number", block.get("block_no")),
        "lines": [],
    }

    for line_index, line in enumerate(block.get("lines", [])):
        line_obj = {
            "id": f"{result['id']}_line{line_index:04d}",
            "bbox": json_safe(line.get("bbox")),
            "wmode": line.get("wmode"),
            "dir": json_safe(line.get("dir")),
            "spans": [],
        }

        for span_index, span in enumerate(line.get("spans", [])):
            span_obj = {
                "id": f"{line_obj['id']}_span{span_index:04d}",
                "bbox": json_safe(span.get("bbox")),
                "origin": json_safe(span.get("origin")),
                "text": normalize_text(span.get("text", "")),
                "font": span.get("font"),
                "font_size": span.get("size"),
                "font_flags": span.get("flags"),
                "font_color": span.get("color"),
                "alpha": span.get("alpha"),
                "ascender": span.get("ascender"),
                "descender": span.get("descender"),
                "char_flags": span.get("char_flags"),
            }
            line_obj["spans"].append(span_obj)

        line_obj["text"] = "".join(s["text"] for s in line_obj["spans"])
        result["lines"].append(line_obj)

    result["text"] = "\n".join(line["text"] for line in result["lines"])
    return result


def extract_page_blocks(page, page_number):
    # "dict" gives a structured representation with text/image blocks.
    raw = page.get_text("dict", sort=False)
    blocks = []

    for block_index, block in enumerate(raw.get("blocks", []), start=1):
        block_type = block.get("type")

        if block_type == 0:
            blocks.append(
                extract_text_block(block, page_number, block_index)
            )

        elif block_type == 1:
            # Image block occurring on the page.
            image_bytes = block.get("image")
            block_obj = {
                "id": f"p{page_number:04d}_block{block_index:04d}",
                "type": "image",
                "bbox": json_safe(block.get("bbox")),
                "width": block.get("width"),
                "height": block.get("height"),
                "ext": block.get("ext"),
                "colorspace": block.get("colorspace"),
                "xres": block.get("xres"),
                "yres": block.get("yres"),
                "bpc": block.get("bpc"),
                "image_hash": sha256_bytes(image_bytes) if image_bytes else None,
                "image_base64": (
                    base64.b64encode(image_bytes).decode("ascii")
                    if image_bytes else None
                ),
            }
            blocks.append(block_obj)

        else:
            # Preserve unknown/raw block types rather than dropping them.
            block_obj = {
                "id": f"p{page_number:04d}_block{block_index:04d}",
                "type": f"pdf_block_type_{block_type}",
                "raw": json_safe(block),
            }
            blocks.append(block_obj)

    return blocks



## 6. Extract drawings / vector graphics

PDFs frequently contain diagrams, lines, rectangles, charts, borders, and other vector objects that are not images.

These are retained separately so the HTML renderer can later decide whether to recreate them as SVG, HTML, or raster images.


In [20]:

def extract_drawings(page):
    drawings = []

    try:
        raw_drawings = page.get_drawings()
    except Exception:
        raw_drawings = []

    for index, drawing in enumerate(raw_drawings, start=1):
        item = {
            "id": f"drawing_{index:05d}",
            "type": "vector_drawing",
            "bbox": json_safe(drawing.get("rect")),
            "items": [],
            "stroke": json_safe(drawing.get("color")),
            "fill": json_safe(drawing.get("fill")),
            "width": drawing.get("width"),
            "line_cap": drawing.get("lineCap"),
            "line_join": drawing.get("lineJoin"),
            "closePath": drawing.get("closePath"),
            "opacity": drawing.get("opacity"),
        }

        for path_item in drawing.get("items", []):
            item["items"].append(json_safe(path_item))

        drawings.append(item)

    return drawings



## 7. Extract links and annotations

Links and annotations are retained because they are part of the document's semantic/interactive content.


In [21]:

def extract_links(page):
    links = []

    try:
        raw_links = page.get_links()
    except Exception:
        raw_links = []

    for index, link in enumerate(raw_links, start=1):
        links.append({
            "id": f"link_{index:05d}",
            "type": "link",
            "raw": json_safe(link),
        })

    return links


def extract_annotations(page):
    annotations = []

    try:
        annot = page.first_annot
        index = 1

        while annot:
            annotations.append({
                "id": f"annot_{index:05d}",
                "type": "annotation",
                "xref": annot.xref,
                "rect": json_safe(annot.rect),
                "info": json_safe(annot.info),
                "vertices": json_safe(annot.vertices),
                "colors": json_safe(annot.colors),
                "opacity": annot.opacity,
                "flags": annot.flags,
            })
            annot = annot.next
            index += 1

    except Exception as exc:
        annotations.append({"error": str(exc)})

    return annotations



## 8. Extract page-level metadata and resources

This includes page size, rotation, crop/media boxes where available, fonts used, images, links, annotations, and a text layer.

The page tree is designed to be the main source for reconstructing HTML.


In [22]:

def extract_page_fonts(page):
    fonts = []

    try:
        for index, font in enumerate(page.get_fonts(full=True), start=1):
            fonts.append({
                "id": f"font_{index:05d}",
                "xref": font[0],
                "name": font[3],
                "full_name": font[3],
                "font_type": font[2],
                "encoding": font[4],
                "subset": font[5] if len(font) > 5 else None,
                "raw": json_safe(font),
            })
    except Exception:
        pass

    return fonts


def extract_page(page, page_number):
    rect = page.rect

    page_obj = {
        "id": f"page_{page_number:04d}",
        "number": page_number,
        "label": page.get_label() if hasattr(page, "get_label") else str(page_number),
        "geometry": {
            "width": float(rect.width),
            "height": float(rect.height),
            "rotation": int(page.rotation),
            "rect": rect_dict(rect),
        },
        "mediabox": json_safe(page.mediabox),
        "cropbox": json_safe(page.cropbox),
        "rotation_matrix": json_safe(page.rotation_matrix),
        "derotation_matrix": json_safe(page.derotation_matrix),
        "content": extract_page_blocks(page, page_number),
        "drawings": extract_drawings(page),
        "links": extract_links(page),
        "annotations": extract_annotations(page),
        "fonts": extract_page_fonts(page),
    }

    # Plain text is retained as a convenience representation.
    page_obj["plain_text"] = normalize_text(page.get_text("text", sort=False))

    # Searchable text statistics.
    page_obj["text_statistics"] = {
        "characters": len(page_obj["plain_text"]),
        "non_whitespace_characters": len(re.sub(r"\s+", "", page_obj["plain_text"])),
        "lines": len(page_obj["plain_text"].splitlines()),
    }

    return page_obj



## 9. Build the complete JSON tree

This is the main extraction step.


In [23]:

def build_document_tree(doc, pdf_path, images):
    pages = []

    for page_number in range(1, len(doc) + 1):
        print(f"Processing page {page_number}/{len(doc)}...")
        pages.append(extract_page(doc[page_number - 1], page_number))

    # Aggregate font information across pages.
    font_usage = Counter()
    for page in pages:
        for font in page["fonts"]:
            key = (font.get("name"), font.get("font_type"))
            font_usage[key] += 1

    fonts = []
    for (name, font_type), count in sorted(font_usage.items()):
        fonts.append({
            "name": name,
            "font_type": font_type,
            "pages_using_font": count,
        })

    # Aggregate statistics.
    text = "\n".join(page["plain_text"] for page in pages)

    statistics = {
        "page_count": len(pages),
        "image_count": len(images),
        "font_count": len(fonts),
        "character_count": len(text),
        "non_whitespace_character_count": len(re.sub(r"\s+", "", text)),
        "word_count": len(re.findall(r"\S+", text)),
        "line_count": len(text.splitlines()),
        "drawing_count": sum(len(p["drawings"]) for p in pages),
        "link_count": sum(len(p["links"]) for p in pages),
        "annotation_count": sum(len(p["annotations"]) for p in pages),
        "text_block_count": sum(
            sum(1 for b in p["content"] if b["type"] == "text")
            for p in pages
        ),
    }

    return {
        "schemaVersion": "1.0.0",
        "schema": "pdf-document-tree",
        "source": {
            "filename": pdf_path.name,
            "extension": pdf_path.suffix.lower(),
            "size_bytes": pdf_path.stat().st_size,
            "sha256": sha256_bytes(pdf_path.read_bytes()),
            "absolute_path": str(pdf_path.resolve()),
        },
        "metadata": document_metadata,
        "statistics": statistics,
        "resources": {
            "images": images,
            "fonts": fonts,
        },
        "pages": pages,
    }


document_tree = build_document_tree(doc, pdf_path, images)

print("\nExtraction complete.")
print(json.dumps(document_tree["statistics"], indent=2))


Processing page 1/4...
Processing page 2/4...
Processing page 3/4...
Processing page 4/4...

Extraction complete.
{
  "page_count": 4,
  "image_count": 2,
  "font_count": 11,
  "character_count": 4819,
  "non_whitespace_character_count": 3877,
  "word_count": 808,
  "line_count": 126,
  "drawing_count": 8,
  "link_count": 0,
  "annotation_count": 0,
  "text_block_count": 72
}



## 10. Inspect the JSON tree

The complete object is available as `document_tree`.

The next cell shows a compact overview rather than printing potentially huge base64 image data.


In [24]:

def redact_binary_for_preview(obj):
    if isinstance(obj, dict):
        result = {}
        for key, value in obj.items():
            if key in {"data_base64", "data_uri", "image_base64"}:
                if isinstance(value, str):
                    result[key] = f"<binary omitted: {len(value):,} characters>"
                else:
                    result[key] = "<binary omitted>"
            else:
                result[key] = redact_binary_for_preview(value)
        return result
    if isinstance(obj, list):
        return [redact_binary_for_preview(v) for v in obj]
    return obj


preview = redact_binary_for_preview(document_tree)

print(json.dumps(preview, indent=2, ensure_ascii=False)[:30000])


{
  "schemaVersion": "1.0.0",
  "schema": "pdf-document-tree",
  "source": {
    "filename": "Statistics.pdf",
    "extension": ".pdf",
    "size_bytes": 302769,
    "sha256": "d63cd7641afa6b45b3ad1d75d8f77b3ea01d24238c8819c982870ee30c2d76f9",
    "absolute_path": "/Users/jonathansanz/LUNA_2/auxiliary/PDF_JS_Jupyter/Statistics.pdf"
  },
  "metadata": {
    "pdf_metadata": {
      "format": "PDF 1.3",
      "title": "",
      "author": "",
      "subject": "",
      "keywords": "",
      "creator": "",
      "producer": "macOS Version 26.5.1 (Build 25F80) Quartz PDFContext",
      "creationDate": "D:20260807103454Z00'00'",
      "modDate": "D:20260807103454Z00'00'",
      "trapped": "",
      "encryption": null
    },
    "page_count": 4,
    "is_encrypted": false,
    "needs_pass": false,
    "permissions": -4,
    "language": null,
    "metadata_extracted_at": "2026-08-15T12:58:30.731798+00:00"
  },
  "statistics": {
    "page_count": 4,
    "image_count": 2,
    "font_count": 11,
   


## 11. Generate a clean JSON file

The JSON file contains the actual binary image data as base64. This makes it self-contained and portable, although it can become large for image-heavy PDFs.

For production, a later version should usually store images separately and keep stable image IDs/paths in the JSON.


In [25]:

output_path = pdf_path.with_name("document_tree.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(document_tree, f, ensure_ascii=False, indent=2)

print(f"Saved: {output_path}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")


Saved: document_tree.json
Size: 1.05 MB



## 12. Optional: save extracted images separately

This is useful for inspecting the extraction and is also closer to how the eventual web application should work.


In [ ]:

image_output_dir = pdf_path.with_name(pdf_path.stem + "_images")
image_output_dir.mkdir(exist_ok=True)

saved = 0

for image in document_tree["resources"]["images"]:
    if "data_base64" not in image:
        continue

    image_bytes = base64.b64decode(image["data_base64"])
    ext = image.get("extension", "bin")
    image_file = image_output_dir / f"{image['id']}.{ext}"

    image_file.write_bytes(image_bytes)
    saved += 1

print(f"Saved {saved} images to: {image_output_dir}")



# What this gives you

This first version intentionally separates **extraction** from **interpretation**.

### Preserved directly from the PDF

- Document metadata
- Page count
- Page dimensions
- Rotation
- MediaBox / CropBox
- Text blocks
- Text lines
- Text spans
- Exact span coordinates
- Font names
- Font sizes
- Font flags
- Text colors
- Character flags
- Images
- Image dimensions
- Image resolution
- Image colorspace
- Image binary data
- Image hashes
- Vector drawings
- Links
- Annotations
- Plain-text representation
- Page-level statistics

### Important next layer

For a high-fidelity PDF → HTML system, the next stage should **not** immediately turn this into HTML.

Instead, build an intermediate **Canonical Document Model (CDM)** that interprets the raw PDF tree into:

```text
document
├── pages
├── sections
├── paragraphs
├── headings
├── lists
├── tables
├── figures
├── equations
├── captions
├── headers
├── footers
└── references
```

while keeping every interpreted element linked back to its original PDF objects and bounding boxes.

That gives you the ability to improve the interpretation logic later without having to re-extract the PDF.
